# Topic 19 - Failure handling: retries, backoff + jitter, Retry-After

Run this from the `implementation/` directory. Everything below uses the
hand-written `failure_core` module with an injected `FakeClock` and `FakeRng`,
so every delay is reproducible.

Scope: retries, timeouts, error taxonomy, idempotency and circuit breaking.
**Provider / model fallbacks live in Topic 19a (fallbacks / failover).**

In [ ]:
import sys

sys.path.insert(0, ".")

from datetime import datetime, timedelta, timezone
from email.utils import format_datetime

from failure_core import (
    ApiError,
    BreakerOpen,
    BreakerState,
    CircuitBreaker,
    ErrorKind,
    FakeClock,
    FakeRng,
    RetryPolicy,
    call_with_breaker,
    classify_error,
    parse_retry_after,
    retry_call,
)

print("failure_core loaded")

## 1. Classify any API error on sight

`classify_error` returns `(ErrorKind, retryable)`. Transient kinds may be
retried; permanent ones must not be.

In [ ]:
samples = [
    ("429 rate limit", ApiError(429, "rate limit exceeded")),
    ("529 overloaded", ApiError(529, "overloaded_error")),
    ("503 server", ApiError(503, "service unavailable")),
    ("408 request timeout", ApiError(408, "request timeout")),
    ("read timeout", TimeoutError("read timeout")),
    ("connection reset", ConnectionError("connection reset by peer")),
    ("409 conflict", ApiError(409, "conflict")),
    ("400 invalid", ApiError(400, "invalid request")),
    ("401 auth", ApiError(401, "invalid api key")),
    ("404 not found", ApiError(404, "model not found")),
]

print(f"{'error':<20} {'kind':<18} retryable")
print("-" * 48)
for label, exc in samples:
    kind, retryable = classify_error(exc)
    print(f"{label:<20} {kind.value:<18} {retryable}")

## 2. Retry with exponential backoff + jitter

`delay = min(base * multiplier ** (attempt - 1), max_delay)`, then jitter is
applied (`full` here: `uniform(0, delay)`). Jitter spreads retries so a fleet of
clients does not retry in lockstep (the thundering herd).

In [ ]:
clock = FakeClock()
rng = FakeRng([1.0, 0.0])   # deterministic jitter samples
calls = []


def flaky():
    calls.append(len(calls) + 1)
    if len(calls) < 3:
        raise ApiError(503, "server error")
    return "answer from attempt 3"


policy = RetryPolicy(max_attempts=5, base=0.5, multiplier=2.0, max_delay=8.0, jitter="full")
outcome = retry_call(flaky, policy, clock, rng)

print("calls       :", calls)
print("delays      :", [round(d, 3) for d in outcome.delays])
print("attempts    :", outcome.attempts)
print("total slept :", round(outcome.total_slept, 3))
print("clock now   :", round(clock.now(), 3))
print("ok / value  :", outcome.ok, "/", outcome.value)

## 3. Retry-After wins over the computed backoff

A 429 usually carries `Retry-After`: either a number of seconds or an HTTP-date.
Prefer the server's value, capped at `max_delay`; header lookup is
case-insensitive.

In [ ]:
retry_after_error = ApiError(429, "slow down", {"retry-after": "3"})


def boom():
    raise retry_after_error


policy = RetryPolicy(max_attempts=3, base=0.5, multiplier=2.0, max_delay=10.0, jitter="none")
outcome = retry_call(boom, policy, FakeClock(), FakeRng())
print("delays        :", outcome.delays)         # server value, not 0.5 / 1.0
print("used header   :", outcome.retry_after_used)

now = datetime(2026, 1, 1, tzinfo=timezone.utc)
print('parse "3"     :', parse_retry_after("3"))
print("parse HTTP-date:", parse_retry_after(format_datetime(now + timedelta(seconds=7)), now=now))

capped = ApiError(503, "overloaded", {"Retry-After": "300"})
outcome = retry_call(
    lambda: (_ for _ in ()).throw(capped),
    RetryPolicy(max_attempts=2, base=0.5, max_delay=2.0, jitter="none"),
    FakeClock(),
    FakeRng(),
)
print("capped at 2.0 :", outcome.delays)

## 4. Circuit breaker: CLOSED -> OPEN -> HALF_OPEN -> CLOSED

Retries live *inside* one request; the breaker spans requests. After the
threshold is hit inside the window it trips OPEN and short-circuits calls until
the cooldown lets one HALF_OPEN trial through.

In [ ]:
clock = FakeClock()
breaker = CircuitBreaker(failure_threshold=3, window=60.0, cooldown=30.0, clock=clock)


def call_api():
    raise ApiError(503, "service unavailable")


print("initial state :", breaker.state.value)
for i in range(3):
    try:
        call_with_breaker(call_api, breaker)
    except ApiError:
        print(f"failure {i + 1}     :", breaker.state.value)

print("allow()       :", breaker.allow())
clock.sleep(30.0)
print("after cooldown:", breaker.allow(), "->", breaker.state.value)
breaker.record_success()
print("after success :", breaker.state.value)

In [ ]:
# An OPEN breaker never touches the dependency.
clock = FakeClock()
breaker = CircuitBreaker(failure_threshold=1, window=60.0, cooldown=30.0, clock=clock)
touched = []


def dependency():
    touched.append(1)
    raise ApiError(503, "down")


try:
    call_with_breaker(dependency, breaker)
except ApiError:
    pass

try:
    call_with_breaker(dependency, breaker)
except BreakerOpen as exc:
    print("short-circuited:", exc)
print("dependency calls:", len(touched), "| state:", breaker.state.value)

## 5. What the SDKs already do vs what is still yours

| Already handled by the SDK | Still your problem |
|---|---|
| ~2 retries, exponential backoff + jitter on 408/409/429/5xx and connection errors | overall request / circuit-level timeout |
| honours `Retry-After` | idempotency keys for write-like calls |
| `max_retries` knob | circuit breaking across requests, budget/spend protection |
| streaming reconnection | retrying only safely-retryable work |

Next topic: **19a - fallbacks / failover** (ordered failover after retries,
attribution for which target served the request).